In [26]:
import pandas as pd
import random
import numpy as np

In [27]:
#Carrega os arquivos CSV
df = pd.read_csv('Metalforte_limpa.csv', sep=',')
df_tecnicos = pd.read_csv('tabela_tecnicos.csv', sep=';', encoding='utf-8-sig')

#Mapeamente das falhas
mapeamento_falhas = {
    'Falha elétrica': ['Eletricista'],
    'Falha mecânica': ['Mecânico'],
    'Lubrificação': ['Automação'],
    'Erro operador': ['Eletricista', 'Mecânico', 'Automação'],
    'Desgaste': ['Automação', 'Mecânico']
}

#Função que busca e sorteia o técnico de forma segura
def alocar_tecnico(causa_falha):
    if pd.isna(causa_falha):
        return pd.Series([None, None, None])
    
    causa_falha = str(causa_falha).strip()
    
    if causa_falha not in mapeamento_falhas:
        return pd.Series([None, None, None])
    
    especialidades_permitidas = mapeamento_falhas[causa_falha]
    
    #Filtra os técnicos do segundo CSV
    tecnicos_validos = df_tecnicos[df_tecnicos['Especialidade'].isin(especialidades_permitidas)]
    
    if tecnicos_validos.empty:
        return pd.Series([None, None, None])
    
    #Sorteia um técnico válido
    tecnico_escolhido = tecnicos_validos.sample(n=1).iloc[0]
    
    #Retorna os dados usando as colunas que agora estão 100% limpas
    return pd.Series([
        tecnico_escolhido['ID_Tecnico'], 
        tecnico_escolhido['Nome_Tecnico'], 
        tecnico_escolhido['Especialidade']
    ])

#Aplica a função criando/atualizando as colunas no DataFrame principal
df[['ID_Tecnico', 'Nome_Tecnico', 'Especialidade_y']] = df['Causa_Falha'].apply(alocar_tecnico)

df.to_excel('Metalforte_final.xlsx', index=False)